# Problem 1(c): MLP for the conditional mean and standard deviation

Network $g:\mathbb{R}^7\to\mathbb{R}^2$ with the link from 1(a), $f_\mu(x)=g_1(x)$ and
$f_\sigma(x)=\exp(g_2(x))$, trained with the Gaussian negative log-likelihood from 1(b).
Compared against two baselines on a held-out test set.

In [1]:
import math
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

SEED = 0
DATA = "../data/taxi_trips.parquet"   # path relative to this notebook (code/)
TARGET = "trip_duration"              # minutes
HALF_LOG_2PI = 0.5 * math.log(2 * math.pi)

## Data

In [2]:
df = pd.read_parquet(DATA)
print(df.shape)
df.head()

(523554, 8)


,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_duration,min_of_day,day_of_week,day_of_year
187173,0.270744,1.425051,0.466831,0.776192,7.965142,-0.942057,-1.047619,1.180855
179655,0.085681,0.306079,-0.226274,0.124111,10.680277,0.778703,-1.047619,-0.311290
120048,-0.555533,-0.934806,-0.022489,-0.548561,5.716184,-1.183016,-1.047619,1.316504
62587,0.069718,0.035804,0.258900,0.317284,4.266709,-0.908008,0.481657,0.018144
212063,-0.234482,0.074616,0.115836,0.258310,11.629685,1.066806,-0.537860,0.521985


In [3]:
df[TARGET].describe()

count    523554.000000
mean         10.413050
std           7.754068
min           0.033386
25%           4.749882
50%           8.631456
75%          13.928152
max         227.942612
Name: trip_duration, dtype: float64

Random 70 / 15 / 15 split into training, validation (for early stopping) and test sets.

In [4]:
X = df.drop(columns=[TARGET]).to_numpy(np.float32)
y = df[TARGET].to_numpy(np.float32)

idx = np.random.default_rng(SEED).permutation(len(df))
n_train, n_val = int(0.70 * len(df)), int(0.15 * len(df))
tr, va, te = np.split(idx, [n_train, n_train + n_val])
X_tr, y_tr = X[tr], y[tr]
X_va, y_va = X[va], y[va]
X_te, y_te = X[te], y[te]
print(f"train {len(y_tr)}, validation {len(y_va)}, test {len(y_te)}")

train 366487, validation 78533, test 78534


## Test metrics (in minutes)

- **NLL**: average Gaussian negative log-likelihood per trip (constant included); scores both $f_\mu$ and $f_\sigma$.
- **RMSE**: accuracy of the mean $f_\mu$ alone.
- **95% coverage**: fraction of test trips inside $f_\mu \pm 1.96 f_\sigma$ (should be close to 0.95).

In [5]:
def gaussian_nll(y, mu, sigma):
    return float(np.mean(np.log(sigma) + 0.5 * ((y - mu) / sigma) ** 2) + HALF_LOG_2PI)

def rmse(y, mu):
    return float(np.sqrt(np.mean((y - mu) ** 2)))

def coverage95(y, mu, sigma):
    return float(np.mean(np.abs(y - mu) <= 1.96 * sigma))

def evaluate(mu, sigma):
    return {"NLL": gaussian_nll(y_te, mu, sigma),
            "RMSE": rmse(y_te, mu),
            "95% coverage": coverage95(y_te, mu, sigma)}

results = {}

## Baselines

1. **Constant**: ignore $x$; predict the training mean and standard deviation for every trip.
2. **Linear mean, constant std**: least-squares linear regression for $f_\mu$, and the training residual
   standard deviation as a constant $f_\sigma$.

In [6]:
mu_const = np.full_like(y_te, y_tr.mean())
sd_const = np.full_like(y_te, y_tr.std())
results["Constant mean and std"] = evaluate(mu_const, sd_const)
results["Constant mean and std"]

{'NLL': 3.4660641111313084,
 'RMSE': 7.745584487915039,
 '95% coverage': 0.95829831665266}

In [7]:
A_tr = np.c_[X_tr, np.ones(len(X_tr))]
coef, *_ = np.linalg.lstsq(A_tr, y_tr, rcond=None)
sd_lin = float(np.std(y_tr - A_tr @ coef))      # residual std on the training set
mu_lin = np.c_[X_te, np.ones(len(X_te))] @ coef
results["Linear mean, constant std"] = evaluate(mu_lin, np.full_like(y_te, sd_lin))
results["Linear mean, constant std"]

{'NLL': 3.4406713913555587,
 'RMSE': 7.551385244641145,
 '95% coverage': 0.9560572490895664}

## MLP

Three hidden layers of width 128 with ReLU, and 2 outputs passed through the 1(a) link.
$g_2$ is clamped to $[-10, 10]$ for numerical safety.

In [8]:
class MeanStdMLP(nn.Module):
    def __init__(self, d_in, d_hidden=128, n_hidden=3):
        super().__init__()
        layers, d = [], d_in
        for _ in range(n_hidden):
            layers += [nn.Linear(d, d_hidden), nn.ReLU()]
            d = d_hidden
        layers.append(nn.Linear(d, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        out = self.net(x)
        mu = out[:, 0]                          # f_mu = g_1 (identity link)
        log_sigma = out[:, 1].clamp(-10, 10)    # f_sigma = exp(g_2)
        return mu, log_sigma


def nll_loss(y, mu, log_sigma):
    # Loss from 1(b): mean of log f_sigma + (y - f_mu)^2 / (2 f_sigma^2), constant dropped
    return (log_sigma + 0.5 * (y - mu) ** 2 * torch.exp(-2 * log_sigma)).mean()

Training: Adam (learning rate $10^{-3}$), batch size 1024, up to 50 epochs, early stopping when the
validation loss has not improved for 5 epochs (the best epoch is kept). Targets are standardized with the
training mean and standard deviation $(\bar y, s)$ for training and converted back afterwards:
$\mu = \bar y + s\,\tilde\mu$, $\sigma = s\,\tilde\sigma$.

In [9]:
torch.manual_seed(SEED)
LR, BATCH, MAX_EPOCHS, PATIENCE = 1e-3, 1024, 50, 5

y_mean, y_std = float(y_tr.mean()), float(y_tr.std())
Xt, yt = torch.from_numpy(X_tr), torch.from_numpy((y_tr - y_mean) / y_std)
Xv, yv = torch.from_numpy(X_va), torch.from_numpy((y_va - y_mean) / y_std)

model = MeanStdMLP(X_tr.shape[1])
opt = torch.optim.Adam(model.parameters(), lr=LR)
history, best_val, best_epoch, best_state, bad = [], math.inf, 0, None, 0
t0 = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    perm = torch.randperm(len(Xt))
    total = 0.0
    for i in range(0, len(Xt), BATCH):
        b = perm[i:i + BATCH]
        loss = nll_loss(yt[b], *model(Xt[b]))
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item() * len(b)

    model.eval()
    with torch.no_grad():
        val_loss = nll_loss(yv, *model(Xv)).item()
    history.append({"epoch": epoch, "train loss": total / len(Xt), "val loss": val_loss})
    print(f"epoch {epoch:2d}   train {total / len(Xt):.4f}   val {val_loss:.4f}")

    if val_loss < best_val - 1e-4:
        best_val, best_epoch, bad = val_loss, epoch, 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        bad += 1
        if bad >= PATIENCE:
            print(f"early stopping: no improvement for {PATIENCE} epochs")
            break

model.load_state_dict(best_state)
model.eval()
print(f"trained in {time.time() - t0:.0f}s; keeping epoch {best_epoch} (val loss {best_val:.4f})")

epoch  1   train -0.0785   val -0.2012
epoch  2   train -0.2150   val -0.2460
epoch  3   train -0.2592   val -0.2675
epoch  4   train -0.2688   val -0.2494
epoch  5   train -0.2864   val -0.2937
epoch  6   train -0.3010   val -0.2910
epoch  7   train -0.3109   val -0.2929
epoch  8   train -0.3187   val -0.2719
epoch  9   train -0.3260   val -0.2973
epoch 10   train -0.3353   val -0.3004
epoch 11   train -0.3437   val -0.2992
epoch 12   train -0.3537   val -0.3124
epoch 13   train -0.3613   val -0.3190
epoch 14   train -0.3640   val -0.3256
epoch 15   train -0.3838   val -0.3114
epoch 16   train -0.3809   val -0.3150
epoch 17   train -0.3949   val -0.3332
epoch 18   train -0.4037   val -0.3347
epoch 19   train -0.3965   val -0.3238
epoch 20   train -0.4089   val -0.3360
epoch 21   train -0.4140   val -0.3413
epoch 22   train -0.4188   val -0.3335
epoch 23   train -0.4229   val -0.3294
epoch 24   train -0.4298   val -0.3291
epoch 25   train -0.3993   val -0.2997
epoch 26   train -0.4257 

## Test-set results

In [10]:
with torch.no_grad():
    mu_s, log_sigma_s = model(torch.from_numpy(X_te))
mu_mlp = y_mean + y_std * mu_s.numpy()       # back to minutes
sd_mlp = y_std * np.exp(log_sigma_s.numpy())

results["MLP (f_mu, f_sigma)"] = evaluate(mu_mlp, sd_mlp)
pd.DataFrame(results).T.round(3)

,NLL,RMSE,95% coverage
Constant mean and std,3.466,7.746,0.958
"Linear mean, constant std",3.441,7.551,0.956
"MLP (f_mu, f_sigma)",2.659,3.812,0.960


Spread of the MLP's predicted standard deviation on the test set (minutes):

In [11]:
pd.Series(sd_mlp, name="predicted sigma").describe(percentiles=[0.05, 0.5, 0.95]).round(2)

count    78534.00
mean         3.49
std          1.73
min          0.77
5%           1.55
50%          3.17
95%          6.42
max         42.39
Name: predicted sigma, dtype: float64

# Problem 1(e): evaluating $P(Y < 45 \mid X = x)$ on the test set

Predicted probability from 1(d): $p_i = \Phi\big((45 - f_\mu(x_i)) / f_\sigma(x_i)\big)$.
True label: $z_i = 1$ if the trip actually took less than 45 minutes.

- **Brier score** $= \frac1n\sum_i (p_i - z_i)^2$ (lower is better)
- **Log-loss** $= -\frac1n\sum_i [z_i \log p_i + (1 - z_i)\log(1 - p_i)]$ (lower is better; $p_i$ clipped to $[10^{-15}, 1 - 10^{-15}]$)
- **Accuracy**: predict "under 45" when $p_i \ge 0.5$, compare with $z_i$

Baselines: a constant probability equal to the training fraction of trips under 45 minutes, and the two 1(c)
baselines passed through the same formula.

In [12]:
from scipy.stats import norm

THRESH = 45.0
z_te = (y_te < THRESH).astype(np.float64)       # true label: trip under 45 minutes
base_rate = float((y_tr < THRESH).mean())       # training fraction under 45 minutes
print(f"fraction of trips under 45 min: train {base_rate:.4f}, test {z_te.mean():.4f}")
print(f"test trips taking 45+ min: {int((1 - z_te).sum())} of {len(z_te)}")

def prob_metrics(p, z=z_te, eps=1e-15):
    p = np.clip(np.asarray(p, dtype=np.float64), eps, 1 - eps)
    return {"Brier": float(np.mean((p - z) ** 2)),
            "Log-loss": float(-np.mean(z * np.log(p) + (1 - z) * np.log(1 - p))),
            "Accuracy": float(np.mean((p >= 0.5) == (z == 1)))}

probs = {
    "Constant (training base rate)": np.full(len(z_te), base_rate),
    "Constant Gaussian (1(c) baseline 1)": norm.cdf(THRESH, loc=mu_const, scale=sd_const),
    "Linear Gaussian (1(c) baseline 2)": norm.cdf(THRESH, loc=mu_lin, scale=sd_lin),
    "MLP from 1(c)": norm.cdf(THRESH, loc=mu_mlp, scale=sd_mlp),
}
pd.DataFrame({k: prob_metrics(v) for k, v in probs.items()}).T.round(4)

fraction of trips under 45 min: train 0.9956, test 0.9955
test trips taking 45+ min: 350 of 78534


,Brier,Log-loss,Accuracy
Constant (training base rate),0.0044,0.0286,0.9955
Constant Gaussian (1(c) baseline 1),0.0045,0.0552,0.9955
Linear Gaussian (1(c) baseline 2),0.0045,0.0466,0.9955
MLP from 1(c),0.0029,0.0223,0.9967


Average predicted probability of "under 45 minutes", split by what actually happened:

In [13]:
long = z_te == 0
pd.DataFrame({k: {"trips under 45": float(v[~long].mean()), "trips 45+": float(v[long].mean())}
              for k, v in probs.items()}).T.round(3)

,trips under 45,trips 45+
Constant (training base rate),0.996,0.996
Constant Gaussian (1(c) baseline 1),1.000,1.000
Linear Gaussian (1(c) baseline 2),1.000,1.000
MLP from 1(c),0.998,0.657
